# Анализ причин отсутствия увеличения выручки маркетплейса

**Задача:** Определить причины отсутствия роста выручки.

**План исследования:**
1. Оценить месячный retention в оформление заказа с помощью когортного анализа.
2. Определить, существует ли product/market fit у этого маркетплейса.
3. Определить 5 основных метрик, на которых продакту можно сконцентрироваться, чтобы максимизировать прибыль компании.
4. Выбрать одну из 3 основных гипотез с помощью фреймворка ICE.
5. Сформулировать нужные метрики, на которые гипотеза должна повлиять.
6. Сделать выводы.

<div style="background-color: #e8f4f8; color: #183153; padding: 16px;">
    <p>Импорт библиотек</p>
</div>

In [1]:
import pandas as pd
import numpy as np
from dateutil import parser
import matplotlib.pylab as plt
import seaborn as sns
from operator import attrgetter

<div style="background-color: #fce4c2; color: #183153; padding: 16px;">
    <b style="font-size: 14px;">1.Оценка месячного retention в оформление заказа с помощью когортного анализа.</b>
</div>

<div style="background-color: #e8f4f8; color: #183153; padding: 16px;">
    <p>1.1 Исследуем данные в таблице <b>olist_orders_dataset.csv</b> и определим, какой вид заказа будем учитывать для анализа retention.</p>
    <b>Описание колонок и значения статусов заказов:</b>
    <ul>
        <li>order_id — уникальный идентификатор заказа (номер чека)</li>
        <li>customer_id — позаказный идентификатор пользователя</li>
        <li>order_status — статус заказа</li>
        <li>order_purchase_timestamp — время создания заказа</li>
        <li>order_approved_at — время подтверждения оплаты заказа</li>
        <li>order_delivered_carrier_date — время передачи заказа в логистическую службу</li>
        <li>order_delivered_customer_date — время доставки заказа</li>
        <li>order_estimated_delivery_date — обещанная дата доставки</li>
    </ul>
    <b>Статусы:</b>
    <ul>
        <li>created — создан</li>
        <li>invoiced — выставлен счёт (и ожидается оплата по нему)</li>
        <li>approved — подтверждён (подтверждена оплата)</li>
        <li>processing — в процессе сборки заказа</li>
        <li>shipped — отгружен со склада</li>
        <li>delivered — доставлен пользователю</li>
        <li>unavailable — недоступен (заказ не выполнен самим магазином; возможно, по причине отсутствия товаров заказа на складе)</li>
        <li>canceled — отменён пользователем</li>
    </ul>
</div>

In [2]:
df_orders = pd.read_csv('olist_orders_dataset.csv', parse_dates=['order_purchase_timestamp',
                                                              'order_approved_at',
                                                              'order_delivered_carrier_date',
                                                              'order_delivered_customer_date',
                                                              'order_estimated_delivery_date'])
print(df_orders.shape)
df_orders.head()

(99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [3]:
# Типы данных в таблице, пропуски. Убедились, что даты преобразованы в datetime

df_orders.info()    

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [4]:
df_orders.isnull().sum()   

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

<div style="background-color: #e8f4f8; color: #183153; padding: 16px;">
    <p>У всех заказов есть статус order_status.</p>
    <p>У 160 заказов нет order_approved_at - времени оплаты заказа.</p>
    <p>У 1783 заказов нет order_delivered_carrier_date - времени передачи заказа в логистическую службу.</p>
    <p>У 2965 заказов нет order_delivered_customer_date - времени доставки заказа.</p>
</div>

In [5]:
# Создадим копию данных, чтобы избежать изменений в исходном датафрейме

orders = df_orders.copy()

In [6]:
# Посмотрим на период времени, который охватывает таблица

# Дата создания самого раннего заказа
print(orders.order_purchase_timestamp.min()) 

# Дата создания самого позднего заказа
print(orders.order_purchase_timestamp.max())

2016-09-04 21:15:19
2018-10-17 17:30:18


In [7]:
# Проверяем наличие дубликатов по order_id (уникальный идентификатор заказа)

orders_duplicated = orders['order_id'].duplicated()
orders_duplicated.sum()

# 0 - дубликатов нет

np.int64(0)

In [8]:
count_by_status = orders.groupby('order_status').agg(count=('order_id', 'nunique'))

# Доля заказов со статусом от общего числа заказов
count_by_status["rate"] = count_by_status["count"].apply(lambda x: x * 100/ 99441)      
count_by_status

,count,rate
order_status,,
approved,2,0.002011
canceled,625,0.628513
created,5,0.005028
delivered,96478,97.020344
invoiced,314,0.315765
processing,301,0.302692
shipped,1107,1.113223
unavailable,609,0.612423


<div style="background-color: #e8f4f8; color: #183153; padding: 16px;">
<p>Посмотрим на распределение заказов по статусам.</p>
<p>Всего заказов - 99441, доставленных - 97% Исходя из распределения статусов заказов и их кол-ва можно предположить, что в таблице указан ПОСЛЕДНИЙ ИЗВЕСТНЫЙ статус заказа.</p>
</div>